In [ ]:
# 1. Install PostgreSQL and driver
!apt-get update -qq > /dev/null
!apt-get install -y postgresql postgresql-contrib > /dev/null
!pip install -q psycopg2-binary

# 2. Start PostgreSQL service
!service postgresql start

# 3. Setup database credentials and privileges
!sudo -u postgres psql -c "CREATE USER taskmaster WITH PASSWORD 'securepass123';"
!sudo -u postgres psql -c "CREATE DATABASE taskdb OWNER taskmaster;"
!sudo -u postgres psql -c "GRANT ALL PRIVILEGES ON DATABASE taskdb TO taskmaster;"

print("PostgreSQL server is running and ready for connections!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/noble/InRelease  503  Service Unavailable [IP: 185.125.189.188 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.
[ OK ]
ERROR:  role "taskmaster" already exists
ERROR:  database "taskdb" already exists
GRANT
PostgreSQL server is running and ready for connections!


In [ ]:
import psycopg2
from psycopg2.extras import RealDictCursor

DB_CONFIG = {
    "dbname": "taskdb",
    "user": "taskmaster",
    "password": "securepass123",
    "host": "localhost",
    "port": "5432"
}

def init_db():
    schema_sql = """
    -- Users Table (Relational Entity)
    CREATE TABLE IF NOT EXISTS users (
        user_id SERIAL PRIMARY KEY,
        username VARCHAR(50) UNIQUE NOT NULL,
        email VARCHAR(100) UNIQUE NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Tasks Table (Foreign Key relationship)
    CREATE TABLE IF NOT EXISTS tasks (
        task_id SERIAL PRIMARY KEY,
        user_id INT NOT NULL REFERENCES users(user_id) ON DELETE CASCADE,
        title VARCHAR(200) NOT NULL,
        status VARCHAR(20) DEFAULT 'pending' CHECK (status IN ('pending', 'in_progress', 'completed')),
        priority INT DEFAULT 1 CHECK (priority BETWEEN 1 AND 5),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Indexes to optimize JOINs and filtering performance
    CREATE INDEX IF NOT EXISTS idx_tasks_user_id ON tasks(user_id);
    CREATE INDEX IF NOT EXISTS idx_tasks_status ON tasks(status);
    """
    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            cur.execute(schema_sql)
            conn.commit()
            print("Schema, Keys, and Indexes initialized successfully.")

init_db()

Schema, Keys, and Indexes initialized successfully.


In [ ]:
class Task:
    """Domain model representing a single Task entity."""
    def __init__(self, title: str, user_id: int, status: str = "pending", priority: int = 1, task_id: int = None):
        self.task_id = task_id
        self.user_id = user_id
        self.title = title
        self.status = status
        self.priority = priority

    def __str__(self):
        return f"[ID: {self.task_id}] {self.title} | Status: {self.status} | Priority: {self.priority}"


class TaskManager:
    """Manages database persistence, queries, JOINs, and aggregates using psycopg2."""

    def __init__(self, db_config: dict):
        self.db_config = db_config

    def _get_connection(self):
        return psycopg2.connect(**self.db_config)

    def register_user(self, username: str, email: str) -> int:
        """Create a new user or fetch existing user ID."""
        sql = """
        INSERT INTO users (username, email)
        VALUES (%s, %s)
        ON CONFLICT (email) DO UPDATE SET username = EXCLUDED.username
        RETURNING user_id;
        """
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, (username, email))
                user_id = cur.fetchone()[0]
                conn.commit()
                return user_id

    def add_task(self, task: Task) -> Task:
        """Persist a new task to PostgreSQL."""
        sql = """
        INSERT INTO tasks (user_id, title, status, priority)
        VALUES (%s, %s, %s, %s)
        RETURNING task_id;
        """
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, (task.user_id, task.title, task.status, task.priority))
                task.task_id = cur.fetchone()[0]
                conn.commit()
                return task

    def update_task_status(self, task_id: int, status: str) -> bool:
        """Update status for a specific task."""
        sql = "UPDATE tasks SET status = %s WHERE task_id = %s;"
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, (status, task_id))
                conn.commit()
                return cur.rowcount > 0

    def list_user_tasks(self, user_id: int):
        """Retrieve tasks using an INNER JOIN between users and tasks."""
        sql = """
        SELECT
            u.username,
            t.task_id,
            t.title,
            t.status,
            t.priority
        FROM users u
        INNER JOIN tasks t ON u.user_id = t.user_id
        WHERE u.user_id = %s
        ORDER BY t.priority DESC;
        """
        with self._get_connection() as conn:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(sql, (user_id,))
                return cur.fetchall()

    def get_summary_statistics(self):
        """
        Demonstrates Aggregate Functions (COUNT, AVG), GROUP BY, and HAVING.
        Calculates aggregate stats for users with at least 1 task.
        """
        sql = """
        SELECT
            u.username,
            COUNT(t.task_id) AS total_tasks,
            COUNT(CASE WHEN t.status = 'completed' THEN 1 END) AS completed_tasks,
            COUNT(CASE WHEN t.status = 'pending' THEN 1 END) AS pending_tasks,
            ROUND(AVG(t.priority), 2) AS avg_priority
        FROM users u
        LEFT JOIN tasks t ON u.user_id = t.user_id
        GROUP BY u.user_id, u.username
        HAVING COUNT(t.task_id) > 0
        ORDER BY total_tasks DESC;
        """
        with self._get_connection() as conn:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(sql,)
                return cur.fetchall()

In [ ]:
def run_cli_app():
    manager = TaskManager(DB_CONFIG)

    print("==========================================")
    print("     WELCOME TO CLI TASK MANAGER DB       ")
    print("==========================================")

    username = input("Enter your username: ").strip()
    email = input("Enter your email: ").strip()

    current_user_id = manager.register_user(username, email)
    print(f"\nAuthenticated as: {username} (User ID: {current_user_id})\n")

    while True:
        print("\n--- MENU ---")
        print("1. Add New Task")
        print("2. View My Tasks (INNER JOIN)")
        print("3. Update Task Status")
        print("4. View All Users Summary (GROUP BY / HAVING)")
        print("5. Exit")

        choice = input("Select an option (1-5): ").strip()

        if choice == '1':
            title = input("Enter task title: ").strip()
            priority = int(input("Enter priority (1-5): ").strip() or 1)
            new_task = Task(title=title, user_id=current_user_id, priority=priority)
            created = manager.add_task(new_task)
            print(f"Task successfully created: {created}")

        elif choice == '2':
            tasks = manager.list_user_tasks(current_user_id)
            print(f"\n--- Tasks for {username} ---")
            if not tasks:
                print("No tasks found.")
            for t in tasks:
                print(f"[{t['task_id']}] {t['title']} | Status: {t['status']} | Priority: {t['priority']}")

        elif choice == '3':
            task_id = int(input("Enter Task ID to update: ").strip())
            print("Status options: pending, in_progress, completed")
            status = input("Enter new status: ").strip().lower()
            if manager.update_task_status(task_id, status):
                print("Task status updated successfully!")
            else:
                print("Task ID not found.")

        elif choice == '4':
            stats = manager.get_summary_statistics()
            print("\n--- System User Productivity Summary ---")
            for row in stats:
                print(f"User: {row['username']} | Total: {row['total_tasks']} | Completed: {row['completed_tasks']} | Pending: {row['pending_tasks']} | Avg Priority: {row['avg_priority']}")

        elif choice == '5':
            print("Exiting CLI application. Goodbye!")
            break
        else:
            print("Invalid choice, please select 1-5.")

# Run the CLI app in Colab
run_cli_app()

     WELCOME TO CLI TASK MANAGER DB       
Enter your username: rohith
Enter your email: guntururohithsai@gmail.com

Authenticated as: rohith (User ID: 1)


--- MENU ---
1. Add New Task
2. View My Tasks (INNER JOIN)
3. Update Task Status
4. View All Users Summary (GROUP BY / HAVING)
5. Exit
Select an option (1-5): 5
Exiting CLI application. Goodbye!


In [ ]:
import json
import logging
import unittest
from datetime import datetime
from decimal import Decimal

# ==========================================
# 1. COMPREHENSIVE LOGGING CONFIGURATION
# ==========================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("app.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("TaskManagerApp")


# Custom JSON Encoder to handle non-serializable types like Decimal and datetime
class CustomJSONEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Decimal):
            return float(obj)
        if isinstance(obj, datetime):
            return obj.isoformat()
        return super().default(obj)


# ==========================================
# 2. EXTENDED TASK MANAGER WITH LOGGING,
#    ERROR HANDLING & JSON EXPORT
# ==========================================
class RobustTaskManager(TaskManager):
    """
    Extends TaskManager with robust error handling, logging,
    and structured JSON export capabilities.
    """

    def register_user(self, username: str, email: str) -> int:
        logger.info(f"Attempting to register user: {username} ({email})")
        try:
            user_id = super().register_user(username, email)
            logger.info(f"User registered successfully. User ID: {user_id}")
            return user_id
        except psycopg2.Error as e:
            logger.error(f"Database error during user registration: {e}")
            raise RuntimeError(f"Failed to register user due to database error: {e}")

    def add_task(self, task: Task) -> Task:
        logger.info(f"Attempting to add task '{task.title}' for user_id: {task.user_id}")
        try:
            created_task = super().add_task(task)
            logger.info(f"Task created successfully with ID: {created_task.task_id}")
            return created_task
        except psycopg2.Error as e:
            logger.error(f"Database error while adding task: {e}")
            raise RuntimeError(f"Failed to add task: {e}")

    def update_task_status(self, task_id: int, status: str) -> bool:
        valid_statuses = ('pending', 'in_progress', 'completed')
        if status not in valid_statuses:
            logger.warning(f"Invalid status transition attempted: '{status}' for task_id: {task_id}")
            raise ValueError(f"Invalid status '{status}'. Must be one of {valid_statuses}")

        logger.info(f"Updating task_id {task_id} status to '{status}'")
        try:
            success = super().update_task_status(task_id, status)
            if success:
                logger.info(f"Task ID {task_id} status updated successfully.")
            else:
                logger.warning(f"No task found with task_id: {task_id}")
            return success
        except psycopg2.Error as e:
            logger.error(f"Database error while updating task {task_id}: {e}")
            raise RuntimeError(f"Database error during status update: {e}")

    def export_summary_to_json(self, filepath: str = "summary_export.json") -> str:
        """Fetches aggregate summary statistics and writes them to a formatted JSON file."""
        logger.info("Generating structured JSON export for user summary statistics...")
        try:
            stats = self.get_summary_statistics()
            json_data = json.dumps(stats, indent=4, cls=CustomJSONEncoder)

            with open(filepath, "w") as f:
                f.write(json_data)

            logger.info(f"Summary report exported successfully to '{filepath}'")
            return json_data
        except Exception as e:
            logger.error(f"Failed to export summary to JSON: {e}")
            raise RuntimeError(f"JSON export failure: {e}")


# ==========================================
# 3. AUTOMATED UNITTEST SUITE
# ==========================================
class TestTaskManagerSuite(unittest.TestCase):
    """
    Automated test suite verifying persistence, relational constraints,
    aggregations, and JSON export logic against the live PostgreSQL backend.
    """

    @classmethod
    def setUpClass(cls):
        """Initialize database schema before executing tests."""
        init_db()
        cls.manager = RobustTaskManager(DB_CONFIG)

    def test_01_register_user_and_persistence(self):
        """Test user creation and idempotent UPSERT logic."""
        user_id = self.manager.register_user("unittest_user", "unittest@example.com")
        self.assertIsNotNone(user_id)
        self.assertIsInstance(user_id, int)

    def test_02_add_and_retrieve_tasks(self):
        """Test task persistence and INNER JOIN query functionality."""
        user_id = self.manager.register_user("unittest_user", "unittest@example.com")
        task = Task(title="Test Task 1", user_id=user_id, priority=5)

        saved_task = self.manager.add_task(task)
        self.assertIsNotNone(saved_task.task_id)

        tasks = self.manager.list_user_tasks(user_id)
        self.assertGreaterEqual(len(tasks), 1)
        self.assertEqual(tasks[0]['title'], "Test Task 1")

    def test_03_update_status_and_validation(self):
        """Test task status updates and runtime validation error handling."""
        user_id = self.manager.register_user("unittest_user", "unittest@example.com")
        task = self.manager.add_task(Task(title="Status Test Task", user_id=user_id))

        # Test valid status update
        updated = self.manager.update_task_status(task.task_id, "completed")
        self.assertTrue(updated)

        # Test invalid status error handling
        with self.assertRaises(ValueError):
            self.manager.update_task_status(task.task_id, "invalid_status_name")

    def test_04_json_export_structure(self):
        """Test structured JSON generation and file serialization."""
        user_id = self.manager.register_user("unittest_user", "unittest@example.com")
        self.manager.add_task(Task(title="JSON Task", user_id=user_id, priority=4))

        json_output = self.manager.export_summary_to_json("test_export.json")
        parsed_data = json.loads(json_output)

        self.assertIsInstance(parsed_data, list)
        if len(parsed_data) > 0:
            self.assertIn("username", parsed_data[0])
            self.assertIn("total_tasks", parsed_data[0])


# ==========================================
# 4. EXECUTION ENTRY POINT
# ==========================================
if __name__ == "__main__":
    print("\n==========================================")
    print("     RUNNING AUTOMATED UNIT TEST SUITE    ")
    print("==========================================")

    # Run unittests programmatically inside Google Colab
    suite = unittest.TestLoader().loadTestsFromTestCase(TestTaskManagerSuite)
    runner = unittest.TextTestRunner(verbosity=2)
    test_result = runner.run(suite)

    if test_result.wasSuccessful():
        print("\n==========================================")
        print("    RUNNING JSON EXPORT DEMONSTRATION     ")
        print("==========================================")

        robust_mgr = RobustTaskManager(DB_CONFIG)
        export_file = "production_summary.json"
        json_str = robust_mgr.export_summary_to_json(export_file)

        print(f"\nGenerated JSON Output ({export_file}):")
        print(json_str)

test_01_register_user_and_persistence (__main__.TestTaskManagerSuite.test_01_register_user_and_persistence)
Test user creation and idempotent UPSERT logic. ... ok
test_02_add_and_retrieve_tasks (__main__.TestTaskManagerSuite.test_02_add_and_retrieve_tasks)
Test task persistence and INNER JOIN query functionality. ... ok
test_03_update_status_and_validation (__main__.TestTaskManagerSuite.test_03_update_status_and_validation)
Test task status updates and runtime validation error handling. ... WARNING:TaskManagerApp:Invalid status transition attempted: 'invalid_status_name' for task_id: 2
ok
test_04_json_export_structure (__main__.TestTaskManagerSuite.test_04_json_export_structure)
Test structured JSON generation and file serialization. ... 


     RUNNING AUTOMATED UNIT TEST SUITE    
Schema, Keys, and Indexes initialized successfully.


ok

----------------------------------------------------------------------
Ran 4 tests in 0.225s

OK



    RUNNING JSON EXPORT DEMONSTRATION     

Generated JSON Output (production_summary.json):
[
    {
        "username": "unittest_user",
        "total_tasks": 3,
        "completed_tasks": 1,
        "pending_tasks": 2,
        "avg_priority": 3.33
    }
]
